In [1]:
import torch
import torch.nn as nn
import numpy as np


In [ ]:
"""
실습 목표
1) 완전한 Tensor Fusion: Z = ⊗_m z_m,  h = W · Z + b
2) Low-rank weight decomposition(CP/LMF 스타일):
   W ≈ Σ_{i=1..r} ⊗_m w_m^(i)   (출력 차원 d_h를 공유하는 형태)
   - (A) W를 복원(reconstruct)해서 h 계산
   - (B) (옵션) 논문 Eq.(6)/(7)처럼 텐서 Z를 만들지 않고 효율적으로 h 계산

주의:
- 여기서는 "append 1"을 포함해서 TFN(부분집합 상호작용) 철학도 반영합니다.
- 모든 코드는 CPU에서 바로 실행 가능.
"""

In [35]:
torch.manual_seed(0)

# -----------------------------
# 0) 설정: bimodal 예시 (M=2)
# -----------------------------
B = 4          # batch size
d_g = 3        # graph vector dim
d_d = 5        # desc vector dim
d_h = 7        # output dim
r = 4          # low-rank

# unimodal vectors (batch)
g = torch.randn(B, d_g)
d = torch.randn(B, d_d)
print(g.shape)
print(g)

print(d.shape)
print(d)


torch.Size([4, 3])
tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986],
        [ 0.4033,  0.8380, -0.7193],
        [-0.4033, -0.5966,  0.1820]])
torch.Size([4, 5])
tensor([[ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740],
        [-0.6787,  0.9383,  0.4889, -0.6731,  0.8728],
        [ 1.0554,  0.1778, -0.5181, -0.3067, -1.5810],
        [ 1.7066, -0.4462,  0.7440,  1.5210,  3.4105]])


In [36]:
# TFN extension: append 1
ones = torch.ones(B, 1)
g_hat = torch.cat([g, ones], dim=1)   # (B, d_g+1)
d_hat = torch.cat([d, ones], dim=1)   # (B, d_d+1)
print(g_hat.shape)
print(g_hat)

print(d_hat.shape)
print(d_hat)

torch.Size([4, 4])
tensor([[ 1.5410, -0.2934, -2.1788,  1.0000],
        [ 0.5684, -1.0845, -1.3986,  1.0000],
        [ 0.4033,  0.8380, -0.7193,  1.0000],
        [-0.4033, -0.5966,  0.1820,  1.0000]])
torch.Size([4, 6])
tensor([[ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000],
        [-0.6787,  0.9383,  0.4889, -0.6731,  0.8728,  1.0000],
        [ 1.0554,  0.1778, -0.5181, -0.3067, -1.5810,  1.0000],
        [ 1.7066, -0.4462,  0.7440,  1.5210,  3.4105,  1.0000]])


In [37]:
# einsum 테스트
print(g_hat[0,:])
print(d_hat[0,:])
print(torch.matmul(g_hat[0,:].view(-1,1), d_hat[0,:].view(1,-1)))

tensor([ 1.5410, -0.2934, -2.1788,  1.0000])
tensor([ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000])
tensor([[ 0.7213, -0.2430,  2.2247,  0.4100, -0.2681,  1.5410],
        [-0.1374,  0.0463, -0.4236, -0.0781,  0.0510, -0.2934],
        [-1.0199,  0.3436, -3.1454, -0.5797,  0.3790, -2.1788],
        [ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000]])


In [ ]:
# -------------------------------------------------------
# 1) 완전한 Tensor Fusion (explicit Z and W)
# 각 배치에 대해 outer product를 쌓은 것
# -------------------------------------------------------
# Z: outer product => (B, dg1, dd1)
Z = torch.einsum("bi,bj->bij", g_hat, d_hat)
print(Z.shape)
print(Z)

torch.Size([4, 4, 6])
tensor([[[ 0.7213, -0.2430,  2.2247,  0.4100, -0.2681,  1.5410],
         [-0.1374,  0.0463, -0.4236, -0.0781,  0.0510, -0.2934],
         [-1.0199,  0.3436, -3.1454, -0.5797,  0.3790, -2.1788],
         [ 0.4681, -0.1577,  1.4437,  0.2660, -0.1740,  1.0000]],

        [[-0.3858,  0.5333,  0.2779, -0.3826,  0.4961,  0.5684],
         [ 0.7361, -1.0176, -0.5302,  0.7300, -0.9466, -1.0845],
         [ 0.9493, -1.3122, -0.6837,  0.9414, -1.2207, -1.3986],
         [-0.6787,  0.9383,  0.4889, -0.6731,  0.8728,  1.0000]],

        [[ 0.4257,  0.0717, -0.2090, -0.1237, -0.6377,  0.4033],
         [ 0.8844,  0.1490, -0.4342, -0.2570, -1.3249,  0.8380],
         [-0.7591, -0.1279,  0.3726,  0.2206,  1.1371, -0.7193],
         [ 1.0554,  0.1778, -0.5181, -0.3067, -1.5810,  1.0000]],

        [[-0.6884,  0.1800, -0.3001, -0.6135, -1.3756, -0.4033],
         [-1.0182,  0.2662, -0.4439, -0.9075, -2.0348, -0.5966],
         [ 0.3107, -0.0812,  0.1354,  0.2769,  0.6208,  0.1820

In [39]:
# Full Weight + Bias
dg1 = d_g + 1
dd1 = d_d + 1

# W_full: (dg1, dd1, d_h), b: (d_h,)
W_full = torch.randn(dg1, dd1, d_h)     # weight
b = torch.randn(d_h)                    # bias

print(W_full.shape)
print(b.shape)

# W*Z + b
# h_full[b,k] = Σ_i Σ_j W[i,j,k] * Z[b,i,j] + b[k]
h_full = torch.einsum("bij,ijk->bk", Z, W_full) + b
print()
print("h_full shape:", h_full.shape)

torch.Size([4, 6, 7])
torch.Size([7])

h_full shape: torch.Size([4, 7])


In [40]:
# -------------------------------------------------------
# 2) Low-rank weight decomposition (LMF-style)
#    W_lowrank = Σ_{i=1..r} (w_g[i] ⊗ w_d[i])   (output dim 공유)
#    where w_g[i] ∈ R^{dg1 × d_h},  w_d[i] ∈ R^{dd1 × d_h}
# -------------------------------------------------------
w_g = torch.randn(r, dg1, d_h)  # modality-specific factors for g
w_d = torch.randn(r, dd1, d_h)  # modality-specific factors for d

print(w_g.shape)
print(w_d.shape)

torch.Size([4, 4, 7])
torch.Size([4, 6, 7])


In [ ]:
# (A) W를 실제로 복원해서 계산 (비효율적이지만 "decomposition" 확인용)
# W_rec: (dg1, dd1, d_h)
W_rec = torch.zeros(dg1, dd1, d_h)
for i in range(r):
    # outer product over non-shared dims only:
    # (dg1, d_h) and (dd1, d_h) -> (dg1, dd1, d_h)
    W_rec += torch.einsum("ah,bh->abh", w_g[i], w_d[i])

h_rec = torch.einsum("bij,ijk->bk", Z, W_rec) + b

print("W_rec shape:", W_rec.shape)
print("h_rec shape:", h_rec.shape)

W_rec shape: torch.Size([4, 6, 7])
h_rec shape: torch.Size([4, 7])


In [47]:
# (B) 효율적 계산: 텐서 Z와 W를 만들지 않고 Eq.(7) 방식으로
# bimodal Eq.(7) 해석:
#   h = (Σ_i (w_g[i]^T z_g))  ∘  (Σ_i (w_d[i]^T z_d))   ?  
# (주의: 논문은 rank-sum과 hadamard 순서가 바뀐 형태도 제시)
#
# 논문 Eq.(8) 스타일(실무 구현과 유사):
#   먼저 각 모달별로 rank별 projection을 만들고,
#   rank별로 모달 간 hadamard 한 뒤,
#   rank 축을 sum:
#
#   fusion_g[b,i,h] = Σ_a g_hat[b,a] * w_g[i,a,h]
#   fusion_d[b,i,h] = Σ_b d_hat[b,b] * w_d[i,b,h]
#   h_eff[b,h] = Σ_i fusion_g[b,i,h] * fusion_d[b,i,h] + b[h]
fusion_g = torch.einsum("bd,rdh->brh", g_hat, w_g)  # (B, r, d_h)
fusion_d = torch.einsum("bd,rdh->brh", d_hat, w_d)  # (B, r, d_h)
h_eff = (fusion_g * fusion_d).sum(dim=1) + b        # (B, d_h)
print("h_eff shape:", h_eff.shape)


h_eff shape: torch.Size([4, 7])


In [48]:



# -------------------------------------------------------
# 3) 검증: (A) 복원 방식과 (B) 효율 방식이 같은지
# -------------------------------------------------------
max_diff = (h_rec - h_eff).abs().max().item()
print("max |h_rec - h_eff| =", max_diff)

# -------------------------------------------------------
# 4) (옵션) "W_full"을 low-rank로 근사해보고 싶다면?
#    실제로는 W_full을 학습하지 않고 w_g, w_d를 학습하지만,
#    여기서는 감각용으로 W_full과 W_rec의 차이를 봅니다.
# -------------------------------------------------------
approx_error = (W_full - W_rec).pow(2).mean().sqrt().item()
print("RMSE(W_full, W_rec) =", approx_error)

"""
출력 해석
- h_full: 완전한 텐서퓨전(임의 W_full) 결과
- h_rec : low-rank factor로 복원한 W_rec로 계산한 결과
- h_eff : 논문 Eq.(8) 스타일의 효율 계산 결과 (h_rec과 같아야 함)
- max_diff가 1e-6 수준이면 실습이 올바른 것.
"""

max |h_rec - h_eff| = 2.384185791015625e-06
RMSE(W_full, W_rec) = 2.275400161743164


'\n출력 해석\n- h_full: 완전한 텐서퓨전(임의 W_full) 결과\n- h_rec : low-rank factor로 복원한 W_rec로 계산한 결과\n- h_eff : 논문 Eq.(8) 스타일의 효율 계산 결과 (h_rec과 같아야 함)\n- max_diff가 1e-6 수준이면 실습이 올바른 것.\n'

In [ ]:
# 검증까지 한 번에 돌리는 최소 예시
import torch
torch.manual_seed(0)

B, d_g, d_d, d_h, r = 4, 3, 5, 7, 4
g = torch.randn(B, d_g)
d = torch.randn(B, d_d)

ones = torch.ones(B, 1)
g_hat = torch.cat([g, ones], dim=1)   # (B, dg1)
d_hat = torch.cat([d, ones], dim=1)   # (B, dd1)

dg1, dd1 = d_g+1, d_d+1

# explicit Z
Z = torch.einsum("bi,bj->bij", g_hat, d_hat)  # (B, dg1, dd1)

# low-rank factors
w_g = torch.randn(r, dg1, d_h)
w_d = torch.randn(r, dd1, d_h)
b = torch.randn(d_h)

# (A) reconstruct W then compute
W_rec = torch.zeros(dg1, dd1, d_h)
for i in range(r):
    W_rec += torch.einsum("ah,bh->abh", w_g[i], w_d[i])
h_rec = torch.einsum("bij,ijk->bk", Z, W_rec) + b

# (B) efficient compute (fixed!)
fusion_g = torch.einsum("bd,rdh->brh", g_hat, w_g)
fusion_d = torch.einsum("bd,rdh->brh", d_hat, w_d)
h_eff = (fusion_g * fusion_d).sum(dim=1) + b

print("max |h_rec - h_eff| =", (h_rec - h_eff).abs().max().item())

max |h_rec - h_eff| = 3.814697265625e-06


In [2]:
li = ['smiles', 'use_MORSE_129', 'use_MORSE_161', 'use_GETAWAY_105', 'use_GETAWAY_133', 'use_GETAWAY_025', 'use_RadiusOfGyration', 'use_MORSE_007', 'use_GETAWAY_085', 'use_GETAWAY_263', 'use_GETAWAY_086', 'use_MORSE_194', 'use_GETAWAY_045', 'use_MORSE_065', 'use_MORSE_193', 'use_MORSE_094', 'use_USRCAT_057', 'use_USRCAT_000', 'use_GETAWAY_005', 'use_GETAWAY_093', 'use_GETAWAY_007', 'use_USRCAT_007', 'use_GETAWAY_002', 'use_GETAWAY_257', 'use_USR_006', 'use_USRCAT_054', 'use_GETAWAY_113', 'use_MORSE_068', 'use_MORSE_097', 'use_MORSE_033', 'use_GETAWAY_067', 'use_GETAWAY_107', 'use_GETAWAY_047', 'use_USRCAT_058', 'use_MORSE_222', 'use_USRCAT_055', 'use_MORSE_167', 'use_USRCAT_043', 'use_GETAWAY_046', 'use_GETAWAY_000', 'use_USRCAT_052', 'use_MORSE_135', 'use_GETAWAY_066', 'use_USRCAT_051', 'use_GETAWAY_084']
len(li)

45

In [4]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=6, sci_mode=False)

# -----------------------------
# Toy setup (very small)
# -----------------------------
bs = 1
M  = 2      # descriptor tokens
d_q = 2     # graph embedding dim
d_t = 2     # token dim
K  = 3      # BAN intermediate dim

# graph embedding f_i (bs, d_q)
f = torch.tensor([[1.0, 2.0]])  # b=0

# descriptor tokens y_j (bs, M, d_t)
Y = torch.tensor([[
    [ 3.0, 4.0],   # j=0
    [-1.0, 2.0],   # j=1
]])

# U: (K, d_q), V: (K, d_t)  (we'll do Uf = f @ U^T, Vy = Y @ V^T)
U = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])  # makes Uf = [1,2,3]

V = torch.tensor([
    [ 1.0,  0.0],
    [ 0.0,  1.0],
    [ 1.0, -1.0],
])  # makes Vy1=[3,4,-1], Vy2=[-1,2,-3]

# glimpse-specific p_g in R^K
p_g = torch.tensor([2.0, -1.0, 0.5])

# -----------------------------
# Compute BAN Eq.(9) in 3 ways
# -----------------------------

# 1) Compute projections
Uf = f @ U.T              # (bs, K)
Vy = Y @ V.T              # (bs, M, K)

# (A) "Equation form": p^T((Uf) ⊙ (Vy_j)) for each j
# logits[b,j] = sum_k p[k] * Uf[b,k] * Vy[b,j,k]
logits_eq9 = torch.einsum("bk,bmk,k->bm", Uf, Vy, p_g)  # (bs, M)

# (B) Hadamard then weighted sum over K
had = Uf.unsqueeze(1) * Vy                         # (bs, M, K)
logits_hadamard = torch.einsum("bmk,k->bm", had, p_g)

# (C) Explicit double-check with a Python loop
logits_loop = torch.zeros(bs, M)
for b in range(bs):
    for j in range(M):
        s = 0.0
        for k in range(K):
            s += p_g[k] * Uf[b, k] * Vy[b, j, k]
        logits_loop[b, j] = s

print("Uf (bs,K):\n", Uf)
print("Vy (bs,M,K):\n", Vy)
print("logits_eq9:\n", logits_eq9)
print("logits_hadamard:\n", logits_hadamard)
print("logits_loop:\n", logits_loop)

# softmax over tokens (just to show attention)
attn = F.softmax(logits_eq9, dim=-1)
print("attn softmax:\n", attn)

Uf (bs,K):
 tensor([[1., 2., 3.]])
Vy (bs,M,K):
 tensor([[[ 3.,  4., -1.],
         [-1.,  2., -3.]]])
logits_eq9:
 tensor([[ -3.500000, -10.500000]])
logits_hadamard:
 tensor([[ -3.500000, -10.500000]])
logits_loop:
 tensor([[ -3.500000, -10.500000]])
attn softmax:
 tensor([[    0.999089,     0.000911]])


In [1]:
a = [1, 2]
a.extend([3, 4])
print(a)

[1, 2, 3, 4]


In [2]:
a = [1, 2]
a.append([3, 4])
print(a)

[1, 2, [3, 4]]


In [3]:
li = [1, 2, 4, 6, 7, 9, 10, 11, 13, 15, 19, 20, 21, 25, 26, 27, 28, 29, 31, 34, 36, 37, 38, 39, 40, 41, 44, 45, 48, 50, 51, 54, 56, 60, 64, 66, 67, 68, 69, 71, 72, 74, 77, 80, 83, 84, 85, 86, 87, 91, 92, 93, 94, 95, 96, 97, 99, 100, 101, 102, 104, 105, 107, 110, 111, 114, 115, 121, 122, 126, 128, 129, 130, 131, 133, 136, 137, 141, 142, 143, 144, 147, 148, 151, 154, 156, 157, 163, 164, 166, 167, 173, 175, 178, 181, 184, 186, 192, 193, 194, 197, 199, 200, 202, 205, 206, 208, 211, 212, 214, 215, 217, 218, 219, 221, 222, 226, 227, 229, 231, 233, 238, 239, 240, 246, 247, 248, 250, 251, 252, 255, 257, 258, 259, 260, 261, 262, 263, 265, 267, 269, 270, 271, 272, 276, 279, 280, 281, 282, 288, 293, 295, 296, 298, 299, 300, 303, 304, 307, 310, 311, 312, 313, 316, 318, 321, 326, 327, 329, 335, 336, 338, 339, 342, 345, 347, 348, 350, 351, 352, 356, 360, 361, 364, 365, 367, 368, 372, 374, 375, 376, 378, 379, 381, 382, 388, 405, 408, 409, 412, 414, 417, 420, 421, 423, 426, 430, 434, 435, 436, 439, 441, 442, 443, 444, 445, 448, 450, 451, 454, 457, 458, 460, 463, 465, 466, 468, 469, 470, 471, 472, 474, 479, 484, 485, 487, 488, 491, 493, 494, 495, 496, 498, 499, 501, 503, 504, 507, 515, 516, 517, 518, 521, 523, 526, 527, 528, 531, 536, 537, 539, 540, 543, 544, 545, 546, 547, 548, 552, 553, 557, 558, 560, 561, 562, 563, 565, 566, 567, 569, 570, 571, 573, 576, 577, 580, 581, 582, 583, 584, 585, 588, 589, 591, 594, 596, 597, 598, 600, 604, 607, 608, 609, 613, 614, 615, 616, 618, 619, 621, 623, 626, 627, 631, 632, 634, 637]
len(li)

317

In [4]:
li = [1, 2, 4, 6, 7, 9, 10, 11, 13, 15, 19, 20, 21, 25, 26, 27, 28, 29, 31, 34, 36, 37, 38, 39, 40, 41, 44, 45, 48, 50, 51, 54, 56, 60, 64, 66, 67, 68, 69, 71, 72, 74, 77, 80, 83, 84, 85, 86, 87, 91, 92, 93, 94, 95, 96, 97, 99, 100, 101, 102, 104, 105, 107, 110, 111, 114, 115, 121, 122, 126, 128, 129, 130, 131, 133, 136, 137, 141, 142, 143, 144, 147, 148, 151, 154, 156, 157, 163, 164, 166, 167, 173, 175, 178, 181, 184, 186, 192, 193, 194, 197, 199, 200, 202, 205, 206, 208, 211, 212, 214, 215, 217, 218, 219, 221, 222, 226, 227, 229, 231, 233, 238, 239, 240, 246, 247, 248, 250, 251, 252, 255, 257, 258, 259, 260, 261, 262, 263, 265, 267, 269, 270, 271, 272, 276, 279, 280, 281, 282, 288, 293, 295, 296, 298, 299, 300, 303, 304, 307, 310, 311, 312, 313, 316, 318, 321, 326, 327, 329, 335, 336, 338, 339, 342, 345, 347, 348, 350, 351, 352, 356, 360, 361, 364, 365, 367, 368, 372, 374, 375, 376, 378, 379, 381, 382, 388, 405, 408, 409, 412, 414, 417, 420, 421, 423, 426, 430, 434, 435, 436, 439, 441, 442, 443, 444, 445, 448, 450, 451, 454, 457, 458, 460, 463, 465, 466, 468, 469, 470, 471, 472, 474, 479, 484, 485, 487, 488, 491, 493, 494, 495, 496, 498, 499, 501, 503, 504, 507, 515, 516, 517, 518, 521, 523, 526, 527, 528, 531, 536, 537, 539, 540, 543, 544, 545, 546, 547, 548, 552, 553, 557, 558, 560, 561, 562, 563, 565, 566, 567, 569, 570, 571, 573, 576, 577, 580, 581, 582, 583, 584, 585, 588, 589, 591, 594, 596, 597, 598, 600, 604, 607, 608, 609, 613, 614, 615, 616, 618, 619, 621, 623, 626, 627, 631, 632, 634, 637, 0, 5, 14, 17, 18, 24, 30, 33, 35, 43, 46, 49, 52, 53, 55, 61, 62, 63, 78, 81, 98, 103, 106, 108, 109, 113, 117, 119, 123, 127, 145, 146, 149, 152, 165, 171, 174, 176, 179, 180, 185, 187, 188, 189, 196, 207, 209, 216, 225, 236, 242, 243, 249, 253, 268, 274, 277, 283, 284, 290, 292, 294, 297, 305, 308, 314, 317, 319, 320, 322, 324, 325, 328, 331, 332, 333, 334, 337, 340, 354, 357, 363, 370, 383, 384, 385, 395, 396, 397, 398, 401, 402, 403, 404, 411, 413, 415, 416, 424, 429, 431, 432, 433, 437, 438, 447, 459, 461, 462, 467, 473, 475, 481, 489, 492, 497, 506, 508, 509, 510, 511, 512, 519, 522, 525, 529, 530, 533, 534, 535, 542, 556, 559, 568, 572, 574, 575, 587, 590, 593, 599, 601, 602, 605, 610, 612, 617, 622, 624, 628, 630, 636]
len(li)

469

In [5]:
152+317

469

In [7]:
li = ['MolLogP', 'HallKierAlpha', 'SMR_VSA10', 'VSA_EState6', 'BCUT2D_CHGHI',
       'MinAbsPartialCharge', 'MaxAbsPartialCharge', 'Kappa2', 'PEOE_VSA6',
       'fr_C_O', 'BalabanJ', 'BCUT2D_LOGPLOW', 'BCUT2D_MWLOW',
       'MaxPartialCharge', 'PEOE_VSA7', 'RingCount', 'SlogP_VSA2',
       'FpDensityMorgan1', 'MaxEStateIndex', 'MaxAbsEStateIndex',
       'NumAromaticHeterocycles', 'PEOE_VSA1', 'NumAromaticCarbocycles',
       'fr_benzene', 'fr_Ar_NH', 'NumHeteroatoms', 'fr_Nhpyrrole',
       'EState_VSA8', 'fr_Ar_OH', 'BCUT2D_LOGPHI', 'SlogP_VSA1', 'PEOE_VSA8',
       'EState_VSA5', 'fr_phos_acid', 'fr_phos_ester', 'EState_VSA7',
       'EState_VSA10', 'fr_Al_OH']
len(li)

38

In [8]:
a={'c1ccccc1': [0, 5, 14, 17, 18, 24, 30, 33, 35, 43, 46, 49, 52, 53, 55, 61, 62, 63, 78, 81, 98, 103, 106, 108, 109, 113, 117, 119, 123, 127, 145, 146, 149, 152, 165, 171, 174, 176, 179, 180, 185, 187, 188, 189, 196, 207, 209, 216, 225, 236, 242, 243, 249, 253, 268, 274, 277, 283, 284, 290, 292, 294, 297, 305, 308, 314, 317, 319, 320, 322, 324, 325, 328, 331, 332, 333, 334, 337, 340, 354, 357, 363, 370, 383, 384, 385, 395, 396, 397, 398, 401, 402, 403, 404, 411, 413, 415, 416, 424, 429, 431, 432, 433, 437, 438, 447, 459, 461, 462, 467, 473, 475, 481, 489, 492, 497, 506, 508, 509, 510, 511, 512, 519, 522, 525, 529, 530, 533, 534, 535, 542, 556, 559, 568, 572, 574, 575, 587, 590, 593, 599, 601, 602, 605, 610, 612, 617, 622, 624, 628, 630, 636], '': [1, 2, 4, 6, 7, 9, 10, 11, 13, 15, 19, 20, 21, 25, 26, 27, 28, 29, 31, 34, 36, 37, 38, 39, 40, 41, 44, 45, 48, 50, 51, 54, 56, 60, 64, 66, 67, 68, 69, 71, 72, 74, 77, 80, 83, 84, 85, 86, 87, 91, 92, 93, 94, 95, 96, 97, 99, 100, 101, 102, 104, 105, 107, 110, 111, 114, 115, 121, 122, 126, 128, 129, 130, 131, 133, 136, 137, 141, 142, 143, 144, 147, 148, 151, 154, 156, 157, 163, 164, 166, 167, 173, 175, 178, 181, 184, 186, 192, 193, 194, 197, 199, 200, 202, 205, 206, 208, 211, 212, 214, 215, 217, 218, 219, 221, 222, 226, 227, 229, 231, 233, 238, 239, 240, 246, 247, 248, 250, 251, 252, 255, 257, 258, 259, 260, 261, 262, 263, 265, 267, 269, 270, 271, 272, 276, 279, 280, 281, 282, 288, 293, 295, 296, 298, 299, 300, 303, 304, 307, 310, 311, 312, 313, 316, 318, 321, 326, 327, 329, 335, 336, 338, 339, 342, 345, 347, 348, 350, 351, 352, 356, 360, 361, 364, 365, 367, 368, 372, 374, 375, 376, 378, 379, 381, 382, 388, 405, 408, 409, 412, 414, 417, 420, 421, 423, 426, 430, 434, 435, 436, 439, 441, 442, 443, 444, 445, 448, 450, 451, 454, 457, 458, 460, 463, 465, 466, 468, 469, 470, 471, 472, 474, 479, 484, 485, 487, 488, 491, 493, 494, 495, 496, 498, 499, 501, 503, 504, 507, 515, 516, 517, 518, 521, 523, 526, 527, 528, 531, 536, 537, 539, 540, 543, 544, 545, 546, 547, 548, 552, 553, 557, 558, 560, 561, 562, 563, 565, 566, 567, 569, 570, 571, 573, 576, 577, 580, 581, 582, 583, 584, 585, 588, 589, 591, 594, 596, 597, 598, 600, 604, 607, 608, 609, 613, 614, 615, 616, 618, 619, 621, 623, 626, 627, 631, 632, 634, 637], 'c1cnccn1': [3, 132, 422], 'C1CCCCC1': [8, 266, 275, 285, 377, 392, 393, 480, 578], 'c1ccncc1': [12, 57, 158, 169, 172, 228, 256, 278, 287, 302, 306, 330, 346, 349, 366, 371, 477, 478, 482, 502, 550, 551], 'c1ccc2c(c1)CCC2': [16], 'C1CCCCCC1': [22], 'C1CC1': [23, 139, 389], 'c1ccc(Nc2ccccc2)cc1': [32], 'c1ccc2ccccc2c1': [42, 201, 223, 235, 254, 301, 387, 427, 453, 464, 513, 554, 629], 'C1CNC1': [47], 'c1ncncn1': [58, 195, 579], 'c1ccc2c(c1)Oc1ccccc1O2': [59, 89, 155, 168, 182, 191, 230, 273, 362, 380, 428, 483], 'O=c1cccnn1-c1ccccc1': [65], 'O=C1c2ccccc2C(=O)c2ccccc21': [70, 244, 245, 476, 606], 'C1CNCCN1': [73, 160, 418], 'C1=CC=CCC=C1': [75], 'c1ccc2c(c1)Cc1ccccc1C2': [76], 'c1cc2c3c(cccc3c1)CC2': [79], 'O=c1cc[nH]c(=O)[nH]1': [82, 90, 153, 210, 425, 446, 538, 586], 'c1c[nH]cn1': [88, 150, 555], 'c1ccc2c(c1)CCO2': [112], 'C1COCCN1': [116, 520], 'c1ccc(-c2ccccc2)cc1': [118, 134, 135, 140, 161, 183, 203, 224, 353, 391, 400, 410, 455, 486, 505, 603], 'O=c1[nH]c(=O)c2[nH]cnc2[nH]1': [120], 'C1CCNCC1': [124, 355], 'c1ccc2c(c1)ccc1ccccc12': [125], 'C1CCOCC1': [138, 213, 359], 'C1=CCCCC1': [159, 204, 323, 592], 'O=C1C=CC(=O)C=C1': [162], 'O=c1[nH]c(=O)[nH]c(=O)[nH]1': [170], 'C1CCNC1': [177, 635], 'O=c1[nH]nnc2ccccc12': [190], 'O=C1CCCC1': [198], 'C1=CC2C3C=CC(C3)C2C1': [220], 'C1CCCC1': [232, 407, 500, 532, 549], 'C1CCOC1': [234, 289, 490], 'c1ccc2cc3ccccc3cc2c1': [237], 'C1CC[SH2+2]C1': [241], 'c1ccc2[nH]ccc2c1': [264], 'c1ccsc1': [286, 452], 'C1=CC2CC1C1CCCC21': [291], 'c1cc[nH]c1': [309, 390], 'O=C1C=CCCC1': [315, 564], 'c1ccc(Cc2ccccc2)cc1': [341], 'c1ccc(Oc2ccccc2)cc1': [343], 'O=C1CCCCC1': [344, 358, 399], 'C=C(c1ccccc1)c1ccccc1': [369], 'c1ccc2c(c1)Cc1ccccc1-2': [373], 'c1ccc(Cn2ccnc2)cc1': [386], 'C=C1c2ccccc2CCc2ccccc21': [394], 'O=C(c1ccccc1)c1ccccc1': [406], 'O=C1NC(=O)c2ccccc21': [419, 620], 'O=C1NC(=O)C2CC=CCC12': [440], 'c1ccc2ncccc2c1': [449], 'c1ccc(NCC2CC2)cc1': [456], 'C1CCC1': [514], 'O=S1OCC2C3C=CC(C3)C2CO1': [524], 'C1=CC2CC1C1C3CC(C4OC34)C21': [541], 'c1cc2ccc3cccc4ccc(c1)c2c34': [595], 'c1cncnc1': [611, 633], 'C1=CCCC1': [625], 'C1COCCO1': [638]}
len(a)

63

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


# =========================================================
# 1. Toy dataset 생성
# =========================================================
def make_toy_dataset(batch_size=4, n_2d=6, n_3d=5):
    """
    예시:
      - 2D descriptor: [ring_count, aromaticity, logP_like, TPSA_like, MW_like, branch_index]
      - 3D descriptor: [volume, surface_area, dipole, radius_gyration, conformer_energy]
    실제 값이라기보다 toy dataset이므로 대충 그럴듯한 범위로 생성
    """

    # 2D descriptors: (B, n_2d)
    desc_2d = torch.tensor([
        [1.0, 0.9, 2.1, 0.4, 0.8, 0.2],
        [0.0, 0.1, 0.7, 0.8, 0.3, 0.6],
        [2.0, 0.8, 1.8, 0.5, 0.9, 0.3],
        [1.0, 0.2, 0.5, 0.9, 0.4, 0.7],
    ], dtype=torch.float32)

    # 3D descriptors: (B, n_3d)
    desc_3d = torch.tensor([
        [0.8, 0.7, 0.3, 0.9, 0.2],
        [0.3, 0.4, 0.8, 0.2, 0.7],
        [0.9, 0.8, 0.4, 0.7, 0.3],
        [0.4, 0.5, 0.7, 0.3, 0.8],
    ], dtype=torch.float32)

    return desc_2d[:batch_size], desc_3d[:batch_size]


# =========================================================
# 2. Descriptor tokenizer
#    scalar descriptor -> token embedding
# =========================================================
class DescriptorTokenizer(nn.Module):
    """
    각 scalar descriptor를 하나의 token으로 바꾼다.
    descriptor value embedding + descriptor id embedding
    출력 shape: (B, num_desc, d_model)
    """
    def __init__(self, num_desc: int, d_model: int):
        super().__init__()
        self.num_desc = num_desc
        self.d_model = d_model

        # descriptor value -> embedding
        self.value_proj = nn.Linear(1, d_model)

        # descriptor id embedding
        self.id_embed = nn.Embedding(num_desc, d_model)

    def forward(self, x):
        """
        x: (B, num_desc)
        """
        B, N = x.shape
        assert N == self.num_desc

        x_val = x.unsqueeze(-1)                           # (B, N, 1)
        val_emb = self.value_proj(x_val)                 # (B, N, d_model)

        ids = torch.arange(N, device=x.device).unsqueeze(0).expand(B, N)  # (B, N)
        id_emb = self.id_embed(ids)                      # (B, N, d_model)

        tokens = val_emb + id_emb
        return tokens


# =========================================================
# 3. Cross attention
#    T2D attends to T3D, and T3D attends to T2D
# =========================================================
class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int = 2, dropout: float = 0.0):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, query_tokens, key_value_tokens):
        """
        query_tokens:     (B, Nq, d_model)
        key_value_tokens: (B, Nk, d_model)

        return:
            out:          (B, Nq, d_model)
            attn_weights: (B, Nq, Nk)
        """
        attn_out, attn_weights = self.cross_attn(
            query=query_tokens,
            key=key_value_tokens,
            value=key_value_tokens,
            need_weights=True,
            average_attn_weights=True
        )
        out = self.norm(query_tokens + attn_out)
        return out, attn_weights


# =========================================================
# 4. Gating block
#    GMU 스타일: h = tanh(Wx), g = sigmoid(Ux), output = g * h
# =========================================================
class GatingBlock(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.h_proj = nn.Linear(d_model, d_model)
        self.g_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: (B, N, d_model)
        return:
          h: transformed feature
          g: gate in (0,1)
          y: gated feature = g * h
        """
        h = torch.tanh(self.h_proj(x))     # (B, N, d_model)
        g = torch.sigmoid(self.g_proj(x))  # (B, N, d_model)
        y = g * h
        return h, g, y


# =========================================================
# 5. Reliability score 계산
#    r = sigmoid(w^T Pool(T_spec))
# =========================================================
class ReliabilityScorer(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.scorer = nn.Linear(d_model, 1)

    def forward(self, x):
        """
        x: (B, N, d_model)
        return:
          pooled: (B, d_model)
          score:  (B, 1) in (0,1)
        """
        pooled = x.mean(dim=1)                 # mean pooling: (B, d_model)
        score = torch.sigmoid(self.scorer(pooled))  # (B, 1)
        return pooled, score


# =========================================================
# 6. 전체 파이프라인
# =========================================================
class ToyFusionModel(nn.Module):
    def __init__(self, n_2d: int, n_3d: int, d_model: int = 8, n_heads: int = 2):
        super().__init__()

        self.tokenizer_2d = DescriptorTokenizer(n_2d, d_model)
        self.tokenizer_3d = DescriptorTokenizer(n_3d, d_model)

        # 1) cross attention
        self.cross_2d_to_3d = CrossAttentionBlock(d_model, n_heads=n_heads)
        self.cross_3d_to_2d = CrossAttentionBlock(d_model, n_heads=n_heads)

        # 2) gating
        self.gate_2d = GatingBlock(d_model)
        self.gate_3d = GatingBlock(d_model)

        # concat(2d,3d) -> gating for shared
        self.shared_proj = nn.Linear(2 * d_model, d_model)
        self.shared_gate = GatingBlock(d_model)

        # 3) score 계산
        self.score_2d = ReliabilityScorer(d_model)
        self.score_3d = ReliabilityScorer(d_model)

    def forward(self, desc_2d, desc_3d):
        """
        desc_2d: (B, n_2d)
        desc_3d: (B, n_3d)
        """
        # ----------------------------------------
        # Tokenize
        # ----------------------------------------
        T2D = self.tokenizer_2d(desc_2d)   # (B, n_2d, d)
        T3D = self.tokenizer_3d(desc_3d)   # (B, n_3d, d)

        # ----------------------------------------
        # 1. Cross attention
        #    T2D <- attend to T3D
        #    T3D <- attend to T2D
        # ----------------------------------------
        T2D_tilde, attn_2d_to_3d = self.cross_2d_to_3d(T2D, T3D)
        T3D_tilde, attn_3d_to_2d = self.cross_3d_to_2d(T3D, T2D)

        # ----------------------------------------
        # 2. Gating for specific
        # ----------------------------------------
        h2d, g2d, T2D_spec = self.gate_2d(T2D_tilde)
        h3d, g3d, T3D_spec = self.gate_3d(T3D_tilde)

        # ----------------------------------------
        # 2. Gating for shared
        # concat(2d,3d) desc -> gating
        #
        # n_2d와 n_3d가 다르므로 여기서는 pooled representation을 써서
        # shared candidate를 만듦
        # ----------------------------------------
        pooled_2d = T2D_tilde.mean(dim=1)                 # (B, d)
        pooled_3d = T3D_tilde.mean(dim=1)                 # (B, d)
        shared_input = torch.cat([pooled_2d, pooled_3d], dim=-1)  # (B, 2d)
        shared_input = self.shared_proj(shared_input)     # (B, d)
        shared_input = shared_input.unsqueeze(1)          # (B, 1, d)

        h_shared, g_shared, T_shared = self.shared_gate(shared_input)  # (B, 1, d)

        # ----------------------------------------
        # 3. score 계산
        # ----------------------------------------
        pooled_spec_2d, r2d = self.score_2d(T2D_spec)     # (B, d), (B,1)
        pooled_spec_3d, r3d = self.score_3d(T3D_spec)     # (B, d), (B,1)

        # ----------------------------------------
        # 4. T_desc 생성
        # T_desc = C(T_shared, r2d*T2D_spec, r3d*T3D_spec)
        # broadcasting 사용
        # ----------------------------------------
        T2D_weighted = r2d.unsqueeze(-1) * T2D_spec       # (B, n_2d, d)
        T3D_weighted = r3d.unsqueeze(-1) * T3D_spec       # (B, n_3d, d)

        T_desc = torch.cat([T_shared, T2D_weighted, T3D_weighted], dim=1)

        return {
            "T2D": T2D,
            "T3D": T3D,
            "T2D_tilde": T2D_tilde,
            "T3D_tilde": T3D_tilde,
            "attn_2d_to_3d": attn_2d_to_3d,
            "attn_3d_to_2d": attn_3d_to_2d,
            "h2d": h2d,
            "g2d": g2d,
            "T2D_spec": T2D_spec,
            "h3d": h3d,
            "g3d": g3d,
            "T3D_spec": T3D_spec,
            "h_shared": h_shared,
            "g_shared": g_shared,
            "T_shared": T_shared,
            "r2d": r2d,
            "r3d": r3d,
            "T_desc": T_desc,
        }


# =========================================================
# 7. 실행 예제
# =========================================================
if __name__ == "__main__":
    # toy dataset
    desc_2d, desc_3d = make_toy_dataset(batch_size=4, n_2d=6, n_3d=5)

    print("desc_2d shape:", desc_2d.shape)   # (4, 6)
    print("desc_3d shape:", desc_3d.shape)   # (4, 5)
    print()

    model = ToyFusionModel(n_2d=6, n_3d=5, d_model=8, n_heads=2)
    out = model(desc_2d, desc_3d)

    # -------------------------
    # shape 확인
    # -------------------------
    print("=== Shape check ===")
    print("T2D       :", out["T2D"].shape)          # (B, 6, 8)
    print("T3D       :", out["T3D"].shape)          # (B, 5, 8)
    print("T2D_tilde :", out["T2D_tilde"].shape)    # (B, 6, 8)
    print("T3D_tilde :", out["T3D_tilde"].shape)    # (B, 5, 8)
    print("T2D_spec  :", out["T2D_spec"].shape)     # (B, 6, 8)
    print("T3D_spec  :", out["T3D_spec"].shape)     # (B, 5, 8)
    print("T_shared  :", out["T_shared"].shape)     # (B, 1, 8)
    print("r2d       :", out["r2d"].shape)          # (B, 1)
    print("r3d       :", out["r3d"].shape)          # (B, 1)
    print("T_desc    :", out["T_desc"].shape)       # (B, 1+6+5, 8)
    print()

    # -------------------------
    # attention weight 확인
    # -------------------------
    print("=== Attention weights ===")
    print("attn_2d_to_3d shape:", out["attn_2d_to_3d"].shape)  # (B, 6, 5)
    print("attn_3d_to_2d shape:", out["attn_3d_to_2d"].shape)  # (B, 5, 6)
    print()

    # 첫 번째 sample의 attention 예시
    print("Sample 0: 2D->3D attention")
    print(out["attn_2d_to_3d"][0])
    print()

    print("Sample 0: 3D->2D attention")
    print(out["attn_3d_to_2d"][0])
    print()

    # -------------------------
    # gating 값 확인
    # -------------------------
    print("=== Gating values ===")
    print("Sample 0: g2d[0] (first 2D token gate)")
    print(out["g2d"][0, 0])  # (8,)
    print()

    print("Sample 0: g3d[0] (first 3D token gate)")
    print(out["g3d"][0, 0])  # (8,)
    print()

    print("Sample 0: g_shared")
    print(out["g_shared"][0, 0])  # (8,)
    print()

    # -------------------------
    # score 확인
    # -------------------------
    print("=== Reliability scores ===")
    print("r2d:")
    print(out["r2d"].squeeze(-1))
    print()

    print("r3d:")
    print(out["r3d"].squeeze(-1))
    print()

    # -------------------------
    # 최종 descriptor representation 확인
    # -------------------------
    print("=== Final T_desc ===")
    print("Sample 0, first 3 tokens of T_desc:")
    print(out["T_desc"][0, :3])   # shared 1개 + 이후 일부 token

desc_2d shape: torch.Size([4, 6])
desc_3d shape: torch.Size([4, 5])

=== Shape check ===
T2D       : torch.Size([4, 6, 8])
T3D       : torch.Size([4, 5, 8])
T2D_tilde : torch.Size([4, 6, 8])
T3D_tilde : torch.Size([4, 5, 8])
T2D_spec  : torch.Size([4, 6, 8])
T3D_spec  : torch.Size([4, 5, 8])
T_shared  : torch.Size([4, 1, 8])
r2d       : torch.Size([4, 1])
r3d       : torch.Size([4, 1])
T_desc    : torch.Size([4, 12, 8])

=== Attention weights ===
attn_2d_to_3d shape: torch.Size([4, 6, 5])
attn_3d_to_2d shape: torch.Size([4, 5, 6])

Sample 0: 2D->3D attention
tensor([[0.1692, 0.3309, 0.1833, 0.1684, 0.1483],
        [0.1565, 0.3570, 0.1410, 0.1969, 0.1486],
        [0.1577, 0.3244, 0.1419, 0.2295, 0.1465],
        [0.2866, 0.1448, 0.1251, 0.2306, 0.2129],
        [0.2077, 0.2059, 0.1510, 0.2302, 0.2053],
        [0.2716, 0.2595, 0.0671, 0.2413, 0.1605]], grad_fn=<SelectBackward0>)

Sample 0: 3D->2D attention
tensor([[0.1280, 0.2276, 0.1766, 0.1853, 0.1151, 0.1674],
        [0.1052, 0.25